In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/housing_cleaned.csv"
)
print(df.shape)

df.head()

(546179, 36)


,unique_id,block_num,unit_count,case_type_name,submission_year,block_address,occupancy_type,occupancy_or_vacancy_date,occupancy_or_vacancy_date_year,bedroom_count,...,data_as_of,data_loaded_at,monthly_rent_mid,square_footage_mid,bedroom_num,bathroom_num,longitude,latitude,longitude.1,latitude.1
0,-8921002025303494677,3640,6.0,Housing Inventory - Unit information (2025),2025,3000 Block of 24TH ST,Occupied by non-owner,NaN,Year Unknown (within past 10-20 years),Two-Bedroom,...,2026-06-24 01:31:02,2026-06-24 06:10:41,1125.5,625.5,2.0,1.0,-122.413262,37.752610,-122.413262,37.752610
1,4926114481077084764,3640,6.0,Housing Inventory - Unit information (2025),2025,3000 Block of 24TH ST,Occupied by non-owner,1999-06-01,1999,Two-Bedroom,...,2026-06-24 01:31:02,2026-06-24 06:10:41,1125.5,625.5,2.0,1.0,-122.413262,37.752610,-122.413262,37.752610
2,-7607276172970613120,3640,6.0,Housing Inventory - Unit information (2025),2025,3000 Block of 24TH ST,Occupied by non-owner,2012-10-06,2012,Two-Bedroom,...,2026-06-24 01:31:02,2026-06-24 06:10:41,1875.5,625.5,2.0,1.0,-122.413262,37.752610,-122.413262,37.752610
3,-3082667924758429561,3643,5.0,Housing Inventory - Unit information (2025),2025,3300 Block of 24TH ST,Occupied by non-owner,2025-06-01,2025,One-Bedroom,...,2026-06-24 01:31:02,2026-06-24 06:10:41,2375.5,625.5,1.0,2.0,-122.420380,37.752176,-122.420380,37.752176
4,8336260795754088004,1711,1.0,Housing Inventory - Unit information (2025),2025,1200 Block of 40TH AVE,Occupied by owner,NaN,NaN,NaN,...,2026-06-24 01:31:02,2026-06-24 06:10:41,NaN,NaN,NaN,NaN,-122.499624,37.763513,-122.499624,37.763513


In [9]:
df.info()

df.describe()

df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546179 entries, 0 to 546178
Data columns (total 36 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   unique_id                            546179 non-null  int64  
 1   block_num                            546173 non-null  object 
 2   unit_count                           546177 non-null  float64
 3   case_type_name                       546179 non-null  object 
 4   submission_year                      546179 non-null  int64  
 5   block_address                        546102 non-null  object 
 6   occupancy_type                       546179 non-null  object 
 7   occupancy_or_vacancy_date            443709 non-null  object 
 8   occupancy_or_vacancy_date_year       477014 non-null  object 
 9   bedroom_count                        516071 non-null  object 
 10  bathroom_count                       516165 non-null  object 
 11  square_footag

unique_id                                   0
block_num                                   6
unit_count                                  2
case_type_name                              0
submission_year                             0
block_address                              77
occupancy_type                              0
occupancy_or_vacancy_date              102470
occupancy_or_vacancy_date_year          69165
bedroom_count                           30108
bathroom_count                          30014
square_footage                            206
monthly_rent                            65920
base_rent_includes_water_sewer              0
base_rent_includes_natural_gas              0
base_rent_includes_electricity              0
base_rent_includes_refuse_recycling         0
base_rent_includes_other_utilities     507856
past_occupancy                         150922
vacancy_date                           525219
signature_date                            134
occupancy_or_vacancy_date_history 

In [24]:
df = df.rename(columns={
    "monthly_rent_mid": "rent",
    "square_footage_mid": "sqft",
    "bedroom_num": "bedrooms",
    "bathroom_num": "bathrooms",
    "analysis_neighborhood": "neighborhood",
})

In [25]:
(df["longitude"] == df["longitude.1"]).all()

(df["latitude"] == df["latitude.1"]).all()

np.False_

In [26]:
df["occupancy_or_vacancy_date"] = pd.to_datetime(
    df["occupancy_or_vacancy_date"],
    errors="coerce"
)

df["signature_date"] = pd.to_datetime(
    df["signature_date"],
    errors="coerce"
)

In [27]:
df["occupancy_or_vacancy_date_year"] = pd.to_numeric(
    df["occupancy_or_vacancy_date_year"],
    errors="coerce"
)

In [28]:
feature_df = df.dropna(
    subset=[
        "rent",
        "sqft",
        "bedrooms",
        "bathrooms",
        "neighborhood"
    ]
).copy()

In [31]:
feature_df["price_per_sqft"] = (
    feature_df["rent"] /
    feature_df["sqft"]
)
feature_df["sqft_per_bedroom"] = (
    feature_df["sqft"] /
    feature_df["bedrooms"]
)
feature_df["bath_bed_ratio"] = (
    feature_df["bathrooms"] /
    feature_df["bedrooms"]
)
feature_df["property_age"] = (
    2026 -
    feature_df["year_property_built"]
)

In [32]:
feature_df["neighborhood_avg_rent"] = (
    feature_df.groupby("neighborhood")["rent"]
              .transform("mean")
)

feature_df["neighborhood_median_rent"] = (
    feature_df.groupby("neighborhood")["rent"]
              .transform("median")
)

feature_df["neighborhood_listing_count"] = (
    feature_df.groupby("neighborhood")["unique_id"]
              .transform("count")
)

feature_df["neighborhood_avg_price_per_sqft"] = (
    feature_df.groupby("neighborhood")["price_per_sqft"]
              .transform("mean")
)

In [33]:
feature_df["rent_difference"] = (
    feature_df["rent"] -
    feature_df["neighborhood_avg_rent"]
)

feature_df["rent_ratio"] = (
    feature_df["rent"] /
    feature_df["neighborhood_avg_rent"]
)

In [34]:
model_df = feature_df[
    [
        "unique_id",
        "rent",
        "sqft",
        "bedrooms",
        "bathrooms",
        "property_age",
        "price_per_sqft",
        "sqft_per_bedroom",
        "bath_bed_ratio",
        "neighborhood",
        "neighborhood_avg_rent",
        "neighborhood_median_rent",
        "neighborhood_avg_price_per_sqft",
        "neighborhood_listing_count",
        "rent_difference",
        "rent_ratio",
        "longitude",
        "latitude",
    ]
]

In [35]:
model_df.to_csv(
    "../data/processed/sf_housing_model_ready.csv",
    index=False
)